# Ligand-Pocket QGNN: Quantum vs Classical Comparison

This notebook compares **Quantum** and **Classical** versions of the Ligand-Pocket QGNN model on the binding classification task.

**Architecture:**
- **Ligand**: Graph Neural Network (GCN) → Latent Vector
- **Pocket**: MLP → Latent Vector
- **Interaction**: Quantum Circuit (VQC) or Classical MLP → Probability

**Task:** Binary Classification (Binding vs Non-Binding)

In [2]:
%load_ext autoreload
%autoreload 2

import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import multiprocessing as mp
from ligand_pocket_qgnn.data import LigandPocketDataProcessor, LigandPocketDataset
from ligand_pocket_qgnn.model import LigandPocketQGNN

# ========== OPTIMIZED COLLATE FUNCTION ==========
def optimized_collate_fn(batch):
    """
    Faster collate function using vectorized PyTorch operations.
    Replaces Python loops with batched tensor operations.
    """
    # Unzip batch
    x_list, edge_index_list, pocket_list, label_list = zip(*batch)

    # Fast concatenation
    pocket_batch = torch.stack(pocket_list)
    label_batch = torch.stack(label_list)

    # Calculate offsets vectorized
    num_nodes_list = torch.tensor([x.shape[0] for x in x_list], dtype=torch.long)
    cumsum = torch.cat([torch.zeros(1, dtype=torch.long), num_nodes_list.cumsum(0)])

    # Batch graphs
    x_batch = torch.cat(x_list, dim=0)

    # Shift edge indices vectorized
    edge_index_shifted = []
    for i, edge_index in enumerate(edge_index_list):
        if edge_index.shape[1] > 0:
            edge_index_shifted.append(edge_index + cumsum[i])

    if edge_index_shifted:
        edge_index_batch = torch.cat(edge_index_shifted, dim=1)
    else:
        edge_index_batch = torch.zeros((2, 0), dtype=torch.long)

    # Create batch vector (which sample each node belongs to)
    batch_vec = torch.cat([torch.full((n,), i, dtype=torch.long)
                           for i, n in enumerate(num_nodes_list)])

    return x_batch, edge_index_batch, batch_vec, pocket_batch, label_batch

print("✓ Optimized collate function loaded!")

✓ Optimized collate function loaded!


## Configuration

In [3]:
# Paths
DATA_DIR = "/media/priyanshu/SD/othercode/data"
SAVE_DIR = "./ligand_pocket_comparison_results"
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. Load Data

In [4]:

# Data parameters
MAX_SAMPLES = 0  # Set to None for full dataset

SEED = 42069
SIMULATION_SEED = 42069  # For quantum simulation
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # OPTIMIZATION: Enable benchmark mode for faster training
    torch.backends.cudnn.deterministic = False  # Set to False for speed
    torch.backends.cudnn.benchmark = True  # Set to True for speed
# Initialize Processor
processor = LigandPocketDataProcessor(DATA_DIR, seed=SEED)

# Load Data
processor.load_data(max_samples=MAX_SAMPLES)

interactions = processor.get_dataset()
print(f"Total Interactions: {len(interactions)}")# Reproducibility

Searching for data in: /media/priyanshu/SD/othercode/data
Found 15239 protein descriptor files


Loading Data: 100%|██████████| 15239/15239 [02:58<00:00, 85.50it/s] 


Generating negative samples (target: 122130)...


Generating Negatives: 100%|██████████| 122130/122130 [30:06<00:00, 67.60it/s]

Loaded 13201 pockets, 122130 ligands
Interactions: 122130 positive, 122130 negative
Total Interactions: 244260


In [5]:


# # Hardware
# print(f"Using device: {DEVICE}")
# CPU_COUNT = mp.cpu_count()

# Model parameters - MUST-RUN EXPERIMENT CONFIGURATION
HIDDEN_DIM = 64
N_QUBITS = 8        
N_QLAYERS = 4      
QUANTUM_DEVICE = 'lightning.gpu'  

# # Training parameters - OPTIMIZED FOR CPU BOTTLENECK
# #BATCH_SIZE = int((GPU_MEMORY_GB * GPU_UTIL_TARGET) * SAMPLES_PER_GB / 12) * 128      # ⭐ DOUBLED: 512 → 1024 (reduce batch frequency)
# NUM_WORKERS = max(4, CPU_COUNT -3)       # ⭐ INCREASED: 8 → 12 (more parallel loading)
PIN_MEMORY = True
# PREFETCH_FACTOR = 4    # ⭐ NEW: Prefetch 4 batches per worker
EPOCHS = 100
LEARNING_RATE_QUANTUM = 0.001
LEARNING_RATE_CLASSICAL = 0.001
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 15

# print(f"\nMUST-RUN EXPERIMENT CONFIGURATION ")
# print(f"{'='*50}")
# print(f"  Random Seed: {SEED}")
# print(f"  Simulation Seed: {SIMULATION_SEED}")
# print(f"  Max Samples: {MAX_SAMPLES}")
# print(f"  Batch Size: {BATCH_SIZE} (INCREASED for GPU)")
# print(f"  Num Workers: {NUM_WORKERS} (INCREASED for CPU)")
# print(f"  Prefetch Factor: {PREFETCH_FACTOR}")
# print(f"  Epochs: {EPOCHS}")
# print(f"  Hidden Dim: {HIDDEN_DIM}")
# print(f"")
# print(f"  Qubits: {N_QUBITS}")
# print(f"  Quantum Layers (Depth): {N_QLAYERS}")
# print(f"  Quantum Device: {QUANTUM_DEVICE}")
# print(f"  Ansatz: StronglyEntanglingLayers")
# print(f"{'='*50}")

In [6]:
# Determine input dimensions from a sample
sample_ligand = processor.ligands[interactions[0].ligand_id]
sample_pocket = processor.pockets[interactions[0].pocket_id]

ligand_dim = sample_ligand.atom_features.shape[1]
pocket_dim = sample_pocket.to_vector().shape[0]

print(f"Ligand Input Dim: {ligand_dim}")
print(f"Pocket Input Dim: {pocket_dim}")


# Auto-detect CPU
CPU_COUNT = mp.cpu_count()
NUM_WORKERS = max(4, CPU_COUNT - 2)
PREFETCH_FACTOR = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


print(f"\n{'='*60}")
print(f"AUTO-TUNING PARAMETERS")
print(f"{'='*60}")
print(f"CPU: {CPU_COUNT} cores → NUM_WORKERS={NUM_WORKERS}")
print(f"PREFETCH_FACTOR: {PREFETCH_FACTOR}")

# Create test model to find optimal batch size
print(f"\nFinding optimal BATCH_SIZE for 12GB VRAM...")
test_model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim,
    pocket_in_dim=pocket_dim,
    hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS,
    n_qlayers=N_QLAYERS,
    use_quantum=True,
    quantum_device=QUANTUM_DEVICE
).to(DEVICE)

# Test increasing batch sizes
BATCH_SIZE = 8192  # Default fallback


# Clean up test model
del test_model
torch.cuda.empty_cache()

# Calculate impact
total_train_samples = len(interactions) * 0.8
batches_per_epoch = int(total_train_samples / BATCH_SIZE)

print(f"\n{'='*60}")
print(f"OPTIMIZED CONFIGURATION")
print(f"{'='*60}")
print(f"CPU:")
print(f"  Cores: {CPU_COUNT}")
print(f"  Workers: {NUM_WORKERS}")
print(f"  Prefetch: {PREFETCH_FACTOR} batches/worker")
print(f"  Total prefetch: {NUM_WORKERS * PREFETCH_FACTOR} batches")
print(f"\nGPU:")
print(f"  Model: {torch.cuda.get_device_name(0)}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  Batch size: {BATCH_SIZE}")
print(f"\nTraining:")
print(f"  Batches/epoch: {batches_per_epoch} (was ~1527)")
print(f"  Speedup: {1527/batches_per_epoch:.1f}x fewer batches")
print(f"  Est. epoch time: {batches_per_epoch * 0.8:.0f}s (~{batches_per_epoch * 0.8 / 60:.1f} min)")
print(f"{'='*60}")

Ligand Input Dim: 10
Pocket Input Dim: 19

AUTO-TUNING PARAMETERS
CPU: 24 cores → NUM_WORKERS=22
PREFETCH_FACTOR: 4

Finding optimal BATCH_SIZE for 12GB VRAM...
✓ Quantum device initialized: lightning.gpu with 8 qubits
  Circuit depth: 4 layers
  Trainable parameters: 96
  Ansatz: StronglyEntanglingLayers

OPTIMIZED CONFIGURATION
CPU:
  Cores: 24
  Workers: 22
  Prefetch: 4 batches/worker
  Total prefetch: 88 batches

GPU:
  Model: NVIDIA GeForce RTX 3080
  VRAM: 12.5 GB
  Batch size: 8192

Training:
  Batches/epoch: 23 (was ~1527)
  Speedup: 66.4x fewer batches
  Est. epoch time: 18s (~0.3 min)


In [7]:
# Create Datasets and Loaders - OPTIMIZED FOR CPU BOTTLENECK
train_ints, val_ints = train_test_split(interactions, test_size=VAL_SPLIT, random_state=SEED)

train_dataset = LigandPocketDataset(processor, train_ints)
val_dataset = LigandPocketDataset(processor, val_ints)

# HEAVILY OPTIMIZED DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=optimized_collate_fn,  
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=optimized_collate_fn,  
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR
)

print(f"✓ DataLoaders created with optimized collate function")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Samples per batch: {BATCH_SIZE}")
print(f"Data loading workers: {NUM_WORKERS}")
print(f"Prefetch factor: {PREFETCH_FACTOR} batches/worker")
print(f"Total prefetched batches: {NUM_WORKERS * PREFETCH_FACTOR}")

✓ DataLoaders created with optimized collate function
Train batches: 24
Val batches: 6
Samples per batch: 8192
Data loading workers: 22
Prefetch factor: 4 batches/worker
Total prefetched batches: 88


In [8]:
# Determine input dimensions from a sample
sample_ligand = processor.ligands[interactions[0].ligand_id]
sample_pocket = processor.pockets[interactions[0].pocket_id]

ligand_dim = sample_ligand.atom_features.shape[1]
pocket_dim = sample_pocket.to_vector().shape[0]

print(f"Ligand Input Dim: {ligand_dim}")
print(f"Pocket Input Dim: {pocket_dim}")

Ligand Input Dim: 10
Pocket Input Dim: 19


## 3. Training Functions

In [9]:
def train_epoch(model, optimizer, criterion, loader, device, show_progress=True):
    """Train one epoch with progress tracking."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training', disable=not show_progress)
    
    for batch_idx, (x_batch, edge_index_batch, batch_vec, pocket_batch, labels) in enumerate(pbar):
        batch_start = datetime.now()
        
        # Non-blocking transfer to GPU
        x_batch = x_batch.to(device, non_blocking=True)
        edge_index_batch = edge_index_batch.to(device, non_blocking=True)
        batch_vec = batch_vec.to(device, non_blocking=True)
        pocket_batch = pocket_batch.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        
        # Forward pass
        outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(outputs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        batch_time = (datetime.now() - batch_start).total_seconds()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'time': f'{batch_time:.1f}s'})
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    
    return {'loss': avg_loss, 'accuracy': accuracy}


def evaluate(model, criterion, loader, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x_batch, edge_index_batch, batch_vec, pocket_batch, labels in tqdm(loader, desc='Validation', leave=False):
            # Non-blocking transfer
            x_batch = x_batch.to(device, non_blocking=True)
            edge_index_batch = edge_index_batch.to(device, non_blocking=True)
            batch_vec = batch_vec.to(device, non_blocking=True)
            pocket_batch = pocket_batch.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds >= 0.5).astype(int)
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds_binary, zero_division=0),
        'recall': recall_score(all_labels, all_preds_binary, zero_division=0),
        'f1': f1_score(all_labels, all_preds_binary, zero_division=0)
    }


def train_model(model, train_loader, val_loader, learning_rate, model_name, device, resume_from_checkpoint=None):
    """Train with detailed progress tracking and optional resume functionality."""
    print(f"\n{'='*70}")
    print(f"Training {model_name.upper()} Model")
    print(f"{'='*70}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Batches per epoch: {len(train_loader)}\n")
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    criterion = nn.BCELoss()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_auc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': [],
        'learning_rates': []
    }
    
    best_val_auc = 0.0
    patience_counter = 0
    start_epoch = 0
    
    # Resume from checkpoint if provided
    if resume_from_checkpoint is not None:
        print(f"{'='*70}")
        print(f"RESUMING FROM CHECKPOINT")
        print(f"{'='*70}")
        
        checkpoint_path = os.path.join(SAVE_DIR, f"{model_name}_best.pt")
        history_path = os.path.join(SAVE_DIR, f"{model_name}_history.json")
        
        if os.path.exists(checkpoint_path) and os.path.exists(history_path):
            # Load checkpoint
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            best_val_auc = checkpoint['best_auc']
            
            # Load history
            with open(history_path, 'r') as f:
                history = json.load(f)
            
            start_epoch = len(history['train_loss'])
            
            # Calculate patience counter (epochs since last improvement)
            best_epoch = np.argmax(history['val_auc'])
            patience_counter = start_epoch - 1 - best_epoch
            
            print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch'] + 1}")
            print(f"✓ Best AUC so far: {best_val_auc:.4f} (epoch {best_epoch + 1})")
            print(f"✓ Resuming from epoch {start_epoch + 1}")
            print(f"✓ Current patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
            
            # Restore scheduler state by running it on past history
            for auc in history['val_auc']:
                scheduler.step(auc)
            
            print(f"✓ Current learning rate: {optimizer.param_groups[0]['lr']:.6f}")
            print(f"{'='*70}\n")
        else:
            print(f"⚠ Checkpoint files not found, starting from scratch")
            print(f"{'='*70}\n")
    
    start_time = datetime.now()
    
    for epoch in range(start_epoch, EPOCHS):
        epoch_start = datetime.now()
        print(f"\n{'='*70}")
        print(f"EPOCH {epoch+1}/{EPOCHS} - Started at {epoch_start.strftime('%H:%M:%S')}")
        print(f"{'='*70}")
        
        train_metrics = train_epoch(model, optimizer, criterion, train_loader, device)
        val_metrics = evaluate(model, criterion, val_loader, device)
        
        # Update scheduler
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_metrics['auc'])
        current_lr = optimizer.param_groups[0]['lr']
        
        if current_lr != old_lr:
            print(f"  Learning rate reduced: {old_lr:.6f} → {current_lr:.6f}")
        
        # Save metrics
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_f1'].append(val_metrics['f1'])
        history['learning_rates'].append(current_lr)
        
        epoch_time = (datetime.now() - epoch_start).total_seconds()
        
        # Print results
        print(f"\n{'-'*70}")
        print(f"Epoch {epoch+1} Results ({epoch_time:.1f}s):")
        print(f"  Train: loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}")
        print(f"  Val:   loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
              f"auc={val_metrics['auc']:.4f}, f1={val_metrics['f1']:.4f}")
        print(f"  Best AUC so far: {best_val_auc:.4f}")
        print(f"{'-'*70}")
        
        # Save best model
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_val_auc,
                'history': history
            }, os.path.join(SAVE_DIR, f"{model_name}_best.pt"))
            
            print(f"✓ New best AUC: {best_val_auc:.4f} (saved)")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
        
        # Save history every epoch
        with open(os.path.join(SAVE_DIR, f"{model_name}_history.json"), 'w') as f:
            json.dump(history, f, indent=2)
        
        # Early stopping
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    total_time = (datetime.now() - start_time).total_seconds()
    print(f"\n{model_name.upper()} Training Complete!")
    print(f"  Total time: {total_time/60:.2f} minutes")
    print(f"  Best AUC: {best_val_auc:.4f}")
    
    return history, best_val_auc

## 4. Train Quantum Model

In [ ]:
print("Creating Quantum Model...")
quantum_model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim,
    pocket_in_dim=pocket_dim,
    hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS,
    n_qlayers=N_QLAYERS,
    use_quantum=True,
    quantum_device=QUANTUM_DEVICE  # Pass the quantum device
)

print(f"\n{'='*70}")
print(f"QUANTUM MODEL ARCHITECTURE")
print(f"{'='*70}")
print(quantum_model)
print(f"{'='*70}")
print(f"Quantum Simulation Seed: {SIMULATION_SEED}")
print(f"{'='*70}")

print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Resume from checkpoint - set to True to continue training
quantum_history, quantum_best_auc = train_model(
    quantum_model,
    train_loader,
    val_loader,
    LEARNING_RATE_QUANTUM,
    "quantum",
    DEVICE,
    resume_from_checkpoint=True  # ← RESUME FROM CHECKPOINT
)

print(f"\nQuantum training finished at: {datetime.now().strftime('%H:%M:%S')}")

Creating Quantum Model...
✓ Quantum device initialized: lightning.gpu with 8 qubits
  Circuit depth: 4 layers
  Trainable parameters: 96
  Ansatz: StronglyEntanglingLayers

QUANTUM MODEL ARCHITECTURE
LigandPocketQGNN(
  (ligand_encoder): LigandGNN(
    (conv1): GCNLayer(
      (linear): Linear(in_features=10, out_features=64, bias=True)
    )
    (conv2): GCNLayer(
      (linear): Linear(in_features=64, out_features=64, bias=True)
    )
    (lin): Linear(in_features=64, out_features=4, bias=True)
  )
  (pocket_encoder): PocketMLP(
    (net): Sequential(
      (0): Linear(in_features=19, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=4, bias=True)
    )
  )
  (interaction): QuantumInteractionLayer(
    (q_layer): <Quantum Torch Layer: func=circuit>
  )
)
Quantum Simulation Seed: 42069

Started at: 2025-12-08 01:07:40


Training QUANTUM Model
Learning Rate: 0.001


Training: 100%|██████████| 24/24 [1:49:41<00:00, 274.22s/it, loss=0.6421, time=236.9s]



----------------------------------------------------------------------
Epoch 28 Results (7671.4s):
  Train: loss=0.6449, acc=0.6177
  Val:   loss=0.6424, acc=0.6234, auc=0.6756, f1=0.5875
  Best AUC so far: 0.6766
----------------------------------------------------------------------
Patience: 3/15

EPOCH 29/100 - Started at 03:15:32


Training: 100%|██████████| 24/24 [1:49:55<00:00, 274.82s/it, loss=0.6510, time=238.8s]



----------------------------------------------------------------------
Epoch 29 Results (7685.4s):
  Train: loss=0.6446, acc=0.6183
  Val:   loss=0.6439, acc=0.6223, auc=0.6745, f1=0.6287
  Best AUC so far: 0.6766
----------------------------------------------------------------------
Patience: 4/15

EPOCH 30/100 - Started at 05:23:37


...

Training: 100%|██████████| 24/24 [1:49:12<00:00, 273.00s/it, loss=0.6413, time=235.6s]



----------------------------------------------------------------------
Epoch 33 Results (7639.9s):
  Train: loss=0.6412, acc=0.6243
  Val:   loss=0.6375, acc=0.6267, auc=0.6831, f1=0.5779
  Best AUC so far: 0.6805
----------------------------------------------------------------------
✓ New best AUC: 0.6831 (saved)

EPOCH 34/100 - Started at 13:53:44


Training: 100%|██████████| 24/24 [1:49:22<00:00, 273.44s/it, loss=0.6409, time=235.2s]



----------------------------------------------------------------------
Epoch 34 Results (7648.0s):
  Train: loss=0.6402, acc=0.6271
  Val:   loss=0.6369, acc=0.6320, auc=0.6832, f1=0.6168
  Best AUC so far: 0.6831
----------------------------------------------------------------------
✓ New best AUC: 0.6832 (saved)

EPOCH 35/100 - Started at 16:01:12


Training:   8%|▊         | 2/24 [09:02<1:39:32, 271.46s/it, loss=0.6472, time=271.8s]

## 5. Train Classical Model

In [ ]:
print("Creating Classical Model...")
classical_model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim,
    pocket_in_dim=pocket_dim,
    hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS,
    n_qlayers=N_QLAYERS,
    use_quantum=False,  # Classical version
)

print(f"\nClassical Model Architecture:")
print(classical_model)

print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

classical_history, classical_best_auc = train_model(
    classical_model,
    train_loader,
    val_loader,
    LEARNING_RATE_CLASSICAL,
    "classical",
    DEVICE
)

print(f"\nClassical training finished at: {datetime.now().strftime('%H:%M:%S')}")

## 6. Comparison Results

In [ ]:
print("\n" + "="*70)
print("FINAL COMPARISON")
print("="*70)
print(f"Quantum Model     - Best AUC: {quantum_best_auc:.4f}")
print(f"Classical Model   - Best AUC: {classical_best_auc:.4f}")
print(f"\nQuantum Advantage: {(quantum_best_auc - classical_best_auc)*100:.2f}% {'improvement' if quantum_best_auc > classical_best_auc else 'deficit'}")
print("="*70)

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss
axes[0, 0].plot(quantum_history['train_loss'], label='Quantum Train', color='blue', alpha=0.7)
axes[0, 0].plot(classical_history['train_loss'], label='Classical Train', color='orange', alpha=0.7)
axes[0, 0].set_title('Training Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[1, 0].plot(quantum_history['val_loss'], label='Quantum Val', color='blue', alpha=0.7)
axes[1, 0].plot(classical_history['val_loss'], label='Classical Val', color='orange', alpha=0.7)
axes[1, 0].set_title('Validation Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(quantum_history['train_acc'], label='Quantum Train', color='blue', alpha=0.7)
axes[0, 1].plot(classical_history['train_acc'], label='Classical Train', color='orange', alpha=0.7)
axes[0, 1].set_title('Training Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 1].plot(quantum_history['val_acc'], label='Quantum Val', color='blue', alpha=0.7)
axes[1, 1].plot(classical_history['val_acc'], label='Classical Val', color='orange', alpha=0.7)
axes[1, 1].set_title('Validation Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# AUC
axes[0, 2].plot(quantum_history['val_auc'], label='Quantum', color='blue', marker='o', alpha=0.7)
axes[0, 2].plot(classical_history['val_auc'], label='Classical', color='orange', marker='s', alpha=0.7)
axes[0, 2].axhline(y=quantum_best_auc, color='blue', linestyle='--', alpha=0.5, label=f'Q Best: {quantum_best_auc:.4f}')
axes[0, 2].axhline(y=classical_best_auc, color='orange', linestyle='--', alpha=0.5, label=f'C Best: {classical_best_auc:.4f}')
axes[0, 2].set_title('Validation AUC')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('AUC')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# F1 Score
axes[1, 2].plot(quantum_history['val_f1'], label='Quantum', color='blue', alpha=0.7)
axes[1, 2].plot(classical_history['val_f1'], label='Classical', color='orange', alpha=0.7)
axes[1, 2].set_title('Validation F1 Score')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('F1')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'comparison_plots.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlots saved to {SAVE_DIR}/comparison_plots.png")

## 8. Final Metrics Summary

In [ ]:
import pandas as pd

# Get best epoch metrics for both models
q_best_idx = np.argmax(quantum_history['val_auc'])
c_best_idx = np.argmax(classical_history['val_auc'])

results_df = pd.DataFrame({
    'Model': ['Quantum', 'Classical'],
    'Best Epoch': [q_best_idx + 1, c_best_idx + 1],
    'Val Loss': [
        quantum_history['val_loss'][q_best_idx],
        classical_history['val_loss'][c_best_idx]
    ],
    'Val Acc': [
        quantum_history['val_acc'][q_best_idx],
        classical_history['val_acc'][c_best_idx]
    ],
    'Val AUC': [quantum_best_auc, classical_best_auc],
    'Val Precision': [
        quantum_history['val_precision'][q_best_idx],
        classical_history['val_precision'][c_best_idx]
    ],
    'Val Recall': [
        quantum_history['val_recall'][q_best_idx],
        classical_history['val_recall'][c_best_idx]
    ],
    'Val F1': [
        quantum_history['val_f1'][q_best_idx],
        classical_history['val_f1'][c_best_idx]
    ]
})

print("\n" + "="*70)
print("SUMMARY OF BEST METRICS")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Save results
results_df.to_csv(os.path.join(SAVE_DIR, 'comparison_results.csv'), index=False)
print(f"\nResults saved to {SAVE_DIR}/comparison_results.csv")